# Metrics and Filtering

## What you'll learn

- Filter artifacts by metric values using a single metric source
- Combine criteria from multiple metric sources into one filter
- Inspect metric values to understand what passed and why

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb)
**Estimated time:** 10 minutes

---

`Filter` routes existing artifacts whose metrics meet your criteria. Run it
first with one metric source, then combine criteria from two sources.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter
from artisan.operations.examples import (
    DataGeneratorWithMetrics,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_metrics,
)

> **Graph legend:** See [Sources and Sequencing](01-sources-and-sequencing.ipynb) for box/arrow key.

## Single-metric filter

`DataGeneratorWithMetrics` produces datasets and associated metrics in one step.
Each dataset gets quality scores like `mean_score` and `std_score`. Filter
auto-discovers these metrics through provenance — you provide the `passthrough`
input (the artifacts to filter) and the criteria.

In [ ]:
env_single = tutorial_setup("single_metric_filter")


def run_single_metric():
    pipeline = PipelineManager.create(
        name="single_metric_filter",
        delta_root=env_single.delta_root,
        staging_root=env_single.staging_root,
        working_root=env_single.working_root,
    )
    output = pipeline.output

    # Generate 5 datasets with co-produced quality metrics
    pipeline.run(
        operation=DataGeneratorWithMetrics,
        name="generate",
        params={"count": 5, "seed": 42},
    )

    # Keep only datasets where mean_score > 0.5
    pipeline.run(
        operation=Filter,
        name="filter",
        inputs={"passthrough": output("generate", "datasets")},
        params={
            "criteria": [
                {"metric": "mean_score", "operator": "gt", "value": 0.5},
            ]
        },
    )

    # Transform the datasets that passed
    pipeline.run(
        operation=DataTransformer,
        name="transform",
        inputs={"dataset": output("filter", "passthrough")},
        params={"seed": 100},
    )
    return pipeline.finalize()


result_single = run_single_metric()

In [ ]:
build_macro_graph(env_single.delta_root)

In [ ]:
build_micro_graph(env_single.delta_root)

### Inspecting the metrics

`inspect_metrics` shows the generated metric values that Filter evaluated.

In [ ]:
inspect_metrics(env_single.delta_root, step_number=0)

Datasets with `mean_score > 0.5` passed to the transform step. Rejected
artifacts remain in the store but are not routed downstream.

Filter follows the dataset-to-metric lineage declared by `DataGeneratorWithMetrics`
to find each dataset's scores. Criteria use bare field names such as `"mean_score"`.
The `"passthrough"` output contains the original artifact IDs.

## Multi-metric filter

Filter can combine metrics produced by different steps. Here,
`DataGeneratorWithMetrics` supplies `mean_score` and `std_score`, while
`MetricCalculator` supplies `distribution.min`, `distribution.range`, and
`summary.cv`. Keep datasets that satisfy both `mean_score > 0.3` and
`distribution.range < 0.9`.

:::{note}
Use dot paths for nested metrics: `"distribution.range"` selects `range` from
the `distribution` group in `MetricCalculator`'s output.
:::

In [ ]:
env_multi = tutorial_setup("multi_metric_filter")


def run_multi_metric():
    pipeline = PipelineManager.create(
        name="multi_metric_filter",
        delta_root=env_multi.delta_root,
        staging_root=env_multi.staging_root,
        working_root=env_multi.working_root,
    )
    output = pipeline.output

    # Generate datasets + central-tendency metrics (mean, std)
    pipeline.run(
        operation=DataGeneratorWithMetrics,
        name="generate",
        params={"count": 5, "seed": 42},
    )

    # Compute distribution metrics (min, max, median, range, CV)
    pipeline.run(
        operation=MetricCalculator,
        name="score",
        inputs={"dataset": output("generate", "datasets")},
    )

    # Filter on criteria from BOTH metric streams
    # Forward provenance walk discovers metrics from both sources automatically.
    # Field names are unique across sources, so no disambiguation needed.
    pipeline.run(
        operation=Filter,
        name="filter",
        inputs={"passthrough": output("generate", "datasets")},
        params={
            "criteria": [
                {"metric": "mean_score", "operator": "gt", "value": 0.3},
                {
                    "metric": "distribution.range",
                    "operator": "lt",
                    "value": 0.9,
                },
            ]
        },
    )

    # Transform the datasets that passed both criteria
    pipeline.run(
        operation=DataTransformer,
        name="transform",
        inputs={"dataset": output("filter", "passthrough")},
        params={"seed": 200},
    )
    return pipeline.finalize()


result_multi = run_multi_metric()

In [ ]:
inspect_metrics(env_multi.delta_root, step_number=1)

In [ ]:
build_macro_graph(env_multi.delta_root)

In [ ]:
build_micro_graph(env_multi.delta_root)

### Inspecting multi-metric results

With two metric sources, viewing all metrics across the pipeline shows both
central-tendency and distribution stats side by side.

In [ ]:
inspect_metrics(env_multi.delta_root)

An artifact must satisfy every criterion to pass. Here Filter matches each
dataset with its `mean_score` from the generate step and `distribution.range`
from the score step.

These field names are unique across sources. If names collide, add `step` or
`step_number` to select a source:

```python
{"metric": "score", "operator": "gt", "value": 0.8, "step": "calc_quality"}
```

:::{seealso}
When a Filter criterion eliminates every artifact, downstream steps receive
empty inputs and are skipped. Use `passthrough_failures=True` to record failures
while passing every artifact through.
See [Error Handling in Practice](../05-errors-and-control/02-error-visibility.ipynb) for details.
:::

## Next steps

- [Multi-Input Operations](04-multi-input-operations.ipynb) — operations that consume multiple input roles
- [Operations Model](../../concepts/operations-model.md) — How creators produce artifacts and curators route them
- [Provenance System](../../concepts/provenance-system.md) — the forward provenance walk that powers metric discovery